# BlackoutLab + ThreadCourt — live Ollama validation
Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. This notebook uses a local Ollama daemon inside your Colab runtime, not paid model APIs. Servers bind to loopback and are reached through Colab's authenticated port proxy. Runtime files and links expire when Colab disconnects; download the evidence bundle before ending the session.

These are authored toy laboratories. AI plans can fail; the deterministic checks own the verdict. One trial per scenario is a demonstration, not a comparative benchmark. Browser/visual QA is separate from Python and DOM integration tests.


In [ ]:
import subprocess, sys, pathlib, os, time, json, urllib.request, shutil
root=pathlib.Path('/content/incident-labs'); root.mkdir(exist_ok=True)
SOURCE_REF='fix/ollama-colab-validation'
def run(args, **kw):
    result=subprocess.run(args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,**kw)
    print(result.stdout,flush=True); result.check_returncode(); return result
run(['nvidia-smi'])
for name in ['BlackoutLab','ThreadCourt']:
    if not (root/name).exists():
        run(['git','clone','--branch',SOURCE_REF,'https://github.com/MinaIbrahim10/'+name+'.git',str(root/name)])
    run(['git','rev-parse','HEAD'],cwd=root/name)
run([sys.executable,'-m','pip','install','-q','-e',str(root/'BlackoutLab')+'[test]','-e',str(root/'ThreadCourt')])
run([sys.executable,'-m','pytest','-q'],cwd=root/'BlackoutLab')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=root/'ThreadCourt')


## Start Ollama and warm the model
The current Linux installer requires `zstd`. Cold GPU initialization can exceed individual agent request timeouts, so warm the model before measuring experiments. Installation and model downloads use official Ollama endpoints.


In [ ]:
if not shutil.which('zstd'):
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','zstd'])
if not shutil.which('ollama'):
    urllib.request.urlretrieve('https://ollama.com/install.sh','/tmp/ollama-install.sh')
    run(['sh','/tmp/ollama-install.sh'])
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/version',timeout=2)
except Exception:
    ollama_process=subprocess.Popen(['ollama','serve'],env={**os.environ,'OLLAMA_HOST':'127.0.0.1:11434'},stdout=open(root/'ollama.log','w'),stderr=subprocess.STDOUT)
    for attempt in range(60):
        try: urllib.request.urlopen('http://127.0.0.1:11434/api/version',timeout=2); break
        except Exception: time.sleep(1)
    else: raise RuntimeError('Ollama failed to start; inspect ollama.log')
MODEL='gemma4:e4b'
run(['ollama','pull',MODEL])
os.environ.update(BLACKOUT_MODEL=MODEL,THREADCOURT_MODEL=MODEL,OLLAMA_BASE_URL='http://127.0.0.1:11434')
request=urllib.request.Request('http://127.0.0.1:11434/api/chat',data=json.dumps({'model':MODEL,'stream':False,'think':False,'messages':[{'role':'user','content':'Reply with OK'}],'options':{'num_ctx':8192,'num_predict':20}}).encode(),headers={'Content-Type':'application/json'})
with urllib.request.urlopen(request,timeout=600) as response:
    data=json.load(response); print('Warmup:',data.get('done_reason'),data.get('message',{}).get('content'))
run(['ollama','ps'])


## Real inference experiments
No runbook fallback is used in single/multi modes. Inspect agent failures, action failures and invariant counts along with recovery rate.


In [ ]:
results=root/'results'; results.mkdir(exist_ok=True)
for mode in ['runbook','single','multi']:
    path=results/(mode+'.json')
    run([sys.executable,'-m','blackout','benchmark','--mode',mode,'--trials','1','--output',str(path)])
for scenario in ['lost-update','lock-inversion']:
    path=results/(scenario+'.json')
    result=subprocess.run([sys.executable,'-m','threadcourt',scenario,'--llm','--output',str(path)],text=True,capture_output=True,check=True)
    data=json.loads(path.read_text())
    print(scenario, 'before:',data['before']['verdict'],'after:',data['after']['verdict'])
    print([(agent['agent'],agent['status']) for agent in data['agents']])


## Open the dashboards
Port 8080 is occupied by Colab, so BlackoutLab uses **8081**. ThreadCourt displays the measured Gemma report for the selected scenario. BlackoutLab keeps its own persisted run history: launch experiments in its UI. If your browser cannot open Colab proxy links, the reports and CLI experiments remain available; do not treat this as completed visual QA.


In [ ]:
from google.colab import output
from IPython.display import display, HTML
from urllib.parse import urlsplit
for name in ['blackout_server','thread_server']:
    proc=globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate(); proc.wait(timeout=10)
blackout_url=output.eval_js('google.colab.kernel.proxyPort(8081)')
threadcourt_url=output.eval_js('google.colab.kernel.proxyPort(8765)')
server_env={**os.environ,'BLACKOUT_ALLOWED_HOSTS':'localhost,127.0.0.1,'+urlsplit(blackout_url).hostname}
blackout_server=subprocess.Popen([sys.executable,'-m','blackout','serve','--port','8081'],cwd=root/'BlackoutLab',env=server_env,stdout=open(root/'blackout-server.log','w'),stderr=subprocess.STDOUT)
selected_scenario='lost-update' # change to lock-inversion to display its report
thread_server=subprocess.Popen([sys.executable,'-c','import json,sys; from threadcourt.dashboard import serve; serve(json.load(open(sys.argv[1])))',str(results/(selected_scenario+'.json'))],stdout=open(root/'threadcourt-server.log','w'),stderr=subprocess.STDOUT)
for port in [8081,8765]:
    for attempt in range(30):
        try: urllib.request.urlopen('http://127.0.0.1:'+str(port),timeout=2); break
        except Exception: time.sleep(1)
    else: raise RuntimeError('Dashboard failed on '+str(port)+'; inspect server logs')
display(HTML('<p><a target="_blank" href="'+blackout_url+'">Open BlackoutLab</a></p><p><a target="_blank" href="'+threadcourt_url+'">Open ThreadCourt</a></p>'))


## Download measured evidence
This exports JSON reports, not Google credentials or browser sessions.


In [ ]:
import zipfile
from google.colab import files
bundle=root/'lab-evidence.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in results.glob('*.json'): archive.write(path,'results/'+path.name)
files.download(str(bundle))
